In [1]:
#  Load dataset and produce a "data quality report": count of nulls per column, duplicate rows, data type issues, value range anomalies 

In [2]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("dirty_cafe_sales.csv")

# Define custom text placeholders that mask actual missing data
placeholders = ["UNKNOWN", "ERROR", "NAN", "NULL", ""]

print("=== DATA QUALITY REPORT ===\n")

# 1. Nulls and Custom Placeholders per Column
print("--- 1. Gaps & Placeholder Analysis ---")
report_data = []
for col in df.columns:
    standard_nulls = df[col].isna().sum()
    text_placeholders = (
        df[col]
        .astype(str)
        .str.strip()
        .str.upper()
        .isin(placeholders)
        .sum()
        - standard_nulls
    )
    total_unclean = standard_nulls + text_placeholders

    report_data.append(
        {
            "Column": col,
            "Standard Nulls (NaN)": standard_nulls,
            "Text Placeholders": text_placeholders,
            "Total Unclean/Missing": total_unclean,
            "% Unclean": round((total_unclean / len(df)) * 100, 2),
        }
    )
print(pd.DataFrame(report_data).to_string(index=False))

# 2. Duplicate Rows and Primary Key Integrity
print("\n--- 2. Duplicate Analysis ---")
print(f"Total Completely Duplicate Rows: {df.duplicated().sum()}")
print(
    f"Duplicate Transaction IDs (Primary Key): {df['Transaction ID'].duplicated().sum()}"
)

# 3. Data Type Issues (Coercion Failures)
print("\n--- 3. Schema Coercion Failures ---")
# Count entries that cannot be converted to their expected semantic types
qty_failures = df["Quantity"].isna().sum() + pd.to_numeric(
    df["Quantity"], errors="coerce"
).isna().sum()
price_failures = df["Price Per Unit"].isna().sum() + pd.to_numeric(
    df["Price Per Unit"], errors="coerce"
).isna().sum()
spent_failures = df["Total Spent"].isna().sum() + pd.to_numeric(
    df["Total Spent"], errors="coerce"
).isna().sum()
date_failures = df["Transaction Date"].isna().sum() + pd.to_datetime(
    df["Transaction Date"], errors="coerce"
).isna().sum()

print(f"Quantity (Expected: Integer) -> Non-convertible rows: {qty_failures}")
print(
    f"Price Per Unit (Expected: Float) -> Non-convertible rows: {price_failures}"
)
print(f"Total Spent (Expected: Float) -> Non-convertible rows: {spent_failures}")
print(
    f"Transaction Date (Expected: Datetime) -> Non-convertible rows: {date_failures}"
)

# 4. Value Range Anomalies
print("\n--- 4. Value Range & Bounds Checking ---")
# Isolate clean numeric arrays to view baseline operational boundaries
clean_qty = pd.to_numeric(df["Quantity"], errors="coerce")
clean_price = pd.to_numeric(df["Price Per Unit"], errors="coerce")
clean_spent = pd.to_numeric(df["Total Spent"], errors="coerce")
clean_date = pd.to_datetime(df["Transaction Date"], errors="coerce")

print(f"Quantity Range:        Min = {clean_qty.min()} | Max = {clean_qty.max()}")
print(
    f"Price Per Unit Range:  Min = ${clean_price.min():.2f} | Max = ${clean_price.max():.2f}"
)
print(
    f"Total Spent Range:     Min = ${clean_spent.min():.2f} | Max = ${clean_spent.max():.2f}"
)
print(
    f"Temporal Boundaries:   Min = {clean_date.min()} | Max = {clean_date.max()}"
)

=== DATA QUALITY REPORT ===

--- 1. Gaps & Placeholder Analysis ---
          Column  Standard Nulls (NaN)  Text Placeholders  Total Unclean/Missing  % Unclean
  Transaction ID                     0                  0                      0       0.00
            Item                   333                303                    636       6.36
        Quantity                   138                203                    341       3.41
  Price Per Unit                   179                175                    354       3.54
     Total Spent                   173                156                    329       3.29
  Payment Method                  2579              -1980                    599       5.99
        Location                  3265              -2569                    696       6.96
Transaction Date                   159                142                    301       3.01

--- 2. Duplicate Analysis ---
Total Completely Duplicate Rows: 0
Duplicate Transaction IDs (Primary Key

In [ ]:
#  Missing data handling: choose an appropriate strategy for each column (mean/median imputation, mode imputation, forward fill, or row deletion) and justify each choice in a markdown cell 

In [5]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("dirty_cafe_sales.csv")

# Standardize text-based missing value placeholders to true NaNs
placeholders = ["UNKNOWN", "ERROR", "NAN", "NULL", ""]
for col in df.columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .mask(df[col].astype(str).str.strip().str.upper().isin(placeholders))
    )

# Force raw types to numeric/date for computational imputation
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")


# =====================================================================
# IMPUTATION PIPELINE
# =====================================================================

# 1. Deterministic Menu Mapping (Hybrid Mode Imputation)
menu_reference = (
    df.dropna(subset=["Item", "Price Per Unit"])
    .drop_duplicates(subset=["Item"])
    .set_index("Price Per Unit")["Item"]
    .to_dict()
)
price_reference = (
    df.dropna(subset=["Item", "Price Per Unit"])
    .drop_duplicates(subset=["Item"])
    .set_index("Item")["Price Per Unit"]
    .to_dict()
)

df["Item"] = df["Item"].fillna(df["Price Per Unit"].map(menu_reference))
df["Price Per Unit"] = df["Price Per Unit"].fillna(
    df["Item"].map(price_reference)
)

# 2. Median Imputation
median_qty = df["Quantity"].median()
df["Quantity"] = df["Quantity"].fillna(median_qty).astype(int)

# 3. Mode Imputation
df["Payment Method"] = df["Payment Method"].fillna(
    df["Payment Method"].mode()[0]
)
df["Location"] = df["Location"].fillna(df["Location"].mode()[0])

# 4. Computed Calculation (Mathematical Imputation)
df["Total Spent"] = df["Quantity"] * df["Price Per Unit"]

# 5. Listwise Deletion (Row Deletion)
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")
df = df.dropna(subset=["Item", "Price Per Unit", "Transaction Date"])

# Final Type Adjustments
df = df.astype({"Quantity": "int32", "Price Per Unit": "float64", "Total Spent": "float64"})

In [6]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,Digital Wallet,Takeaway,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9994,TXN_7851634,Sandwich,4,4.0,16.0,Digital Wallet,Takeaway,2023-01-08
9995,TXN_7672686,Coffee,2,2.0,4.0,Digital Wallet,Takeaway,2023-08-30
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,Takeaway,2023-03-02
9998,TXN_7695629,Cookie,3,1.0,3.0,Digital Wallet,Takeaway,2023-12-02


In [ ]:
|### Missing Data Imputation & Handling Strategy

To ensure analytical integrity without sacrificing valuable transactional volume, we apply tailored statistical and logical rules to resolve missing values rather than relying on blunt row deletion.

| Column Name | Strategy Selected | Justified Reason & Business Context |
| :--- | :--- | :--- |
| **Transaction ID** | **No Action / Row Deletion** | As the primary key, it contains 0% missing values. If a row ever lacked a `Transaction ID`, it would face immediate row deletion because an unrecoverable primary audit trail cannot be synthetically generated. |
| **Item** | **Hybrid Mode Imputation (Relational Mapping)** | Dropping these would bias sales volume downward. Because the cafe menu has a deterministic 1:1 relationship between item names and prices (e.g., all `$1.50` items are always `Tea`), we map the mode item for that price point rather than arbitrary imputation. If both price and name are missing, the row is deleted. |
| **Quantity** | **Median Imputation** | Since quantity must be a discrete integer ($1$ to $5$ cups/items), using a mean would yield skewed decimal values ($2.34$). The median ($2.0$) safely preserves the ordinal distribution without introducing unrealistic fractional transactions. |
| **Price Per Unit** | **Hybrid Mode Imputation (Relational Mapping)** | Similar to the `Item` column, a product's price is structurally fixed. Imputing via an overall dataset mean or median would introduce synthetic prices (e.g., a `$3.12` coffee) that do not exist on the menu. We use the exact item menu lookup map to fill missing values. |
| **Total Spent** | **Computed Imputation** | Imputing via mean or median here introduces severe mathematical contradictions against `Quantity` and `Price Per Unit`. Re-calculating the column mathematically via $Quantity \times Price\ Per\ Unit$ guarantees absolute financial accuracy. |
| **Payment Method** | **Mode Imputation** | Since transactional payment data lacks a timeline-based pattern, forward-filling would falsely carry over personal customer attributes. Imputing with the most common method (the mathematical mode) keeps the global categorical distributions intact for macro-channel analysis. |
| **Location** | **Mode Imputation** | Preserves row counts without altering global operational distributions. Forward-fill is avoided because it would imply a consecutive series of orders at the exact same counter style, which physically behaves as a random process. |
| **Transaction Date** | **Row Deletion (Listwise Deletion)** | Time is the structural anchor for time-series analytics. Imputing dates via mean/median creates synthetic temporal artifacts, and forward-filling can disrupt true chronological order. Unparseable dates must be deleted to maintain absolute timeline authenticity. |

In [7]:
# Duplicate removal: identify and remove duplicate rows; document how many were removed 

In [8]:
import pandas as pd

# Load dataset
df = pd.read_csv("dirty_cafe_sales.csv")

print("=== DUPLICATE REMOVAL PROCESS ===")

# 1. Identify exact duplicate rows (all columns matching)
exact_duplicates_count = df.duplicated().sum()
print(f"Identified {exact_duplicates_count} completely identical rows.")

# 2. Identify primary key collisions (Duplicate Transaction IDs with differing data)
pk_duplicates_count = df["Transaction ID"].duplicated().sum()
print(
    f"Identified {pk_duplicates_count} duplicate Transaction IDs (Primary Key collisions)."
)

# 3. Execute removal operations
# Keep the first occurrence of any duplicate row
df_cleaned = df.drop_duplicates(keep="first")

# Deduplicate primary keys if any collisions exist (keeping first entry)
df_cleaned = df_cleaned.drop_duplicates(subset=["Transaction ID"], keep="first")

# Calculate totals removed
total_rows_removed = len(df) - len(df_cleaned)
print(f"\nPipeline Action: Removed {total_rows_removed} total rows.")
print(f"Current working row count: {len(df_cleaned)}")

=== DUPLICATE REMOVAL PROCESS ===
Identified 0 completely identical rows.
Identified 0 duplicate Transaction IDs (Primary Key collisions).

Pipeline Action: Removed 0 total rows.
Current working row count: 10000


In [9]:
# Standardisation: normalise inconsistent formatting (e.g., "Male"/"male"/"M" → "Male"; date formats → datetime) 

In [10]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("dirty_cafe_sales.csv")

print("=== STANDARDIZATION PIPELINE ===\n")

# --- 1. Date Format Standardization ---
print("Executing Temporal Standardization...")
initial_invalid_dates = (
    df["Transaction Date"].isna().sum()
    + pd.to_datetime(df["Transaction Date"], errors="coerce").isna().sum()
)

# Convert strings to robust datetime objects, forcing corrupt patterns to NaT
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

print(
    f"-> Restructured date series to datetime64[ns]. Handled {initial_invalid_dates} invalid strings.\n"
)


# --- 2. Categorical & Text Case Standardization ---
print("Executing Categorical Harmonization...")

# Define structural noisy values to strip out
placeholders = ["UNKNOWN", "ERROR", "NAN", "NULL", ""]

categorical_cols = ["Item", "Payment Method", "Location"]
for col in categorical_cols:
    # Remove leading/trailing whitespaces and synchronize to Title Case
    df[col] = df[col].astype(str).str.strip().str.title()

    # Roll back string representations of missing tokens to true NaNs
    df[col] = df[col].mask(df[col].str.upper().isin(placeholders))

print("-> Standardized text to Clean Title Case (e.g., 'Takeaway', 'Credit Card').")
print("-> Replaced hidden string errors with formal structural nulls.\n")


# --- 3. Numerical Integrity & Recalculation ---
print("Executing Mathematical Alignment...")

# Force data types for numeric evaluation
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")

# Re-calculate Total Spent programmatically to remove text anomalies like "ERROR"
df["Total Spent"] = df["Quantity"] * df["Price Per Unit"]
print("-> Corrected calculation schema: Total Spent = Quantity * Price Per Unit.")

print("\n=== FINAL SCHEMA VALIDATION ===")
print(df[["Transaction Date", "Item", "Payment Method", "Total Spent"]].head(10))

=== STANDARDIZATION PIPELINE ===

Executing Temporal Standardization...
-> Restructured date series to datetime64[ns]. Handled 619 invalid strings.

Executing Categorical Harmonization...
-> Standardized text to Clean Title Case (e.g., 'Takeaway', 'Credit Card').
-> Replaced hidden string errors with formal structural nulls.

Executing Mathematical Alignment...
-> Corrected calculation schema: Total Spent = Quantity * Price Per Unit.

=== FINAL SCHEMA VALIDATION ===
  Transaction Date      Item  Payment Method  Total Spent
0       2023-09-08    Coffee     Credit Card          4.0
1       2023-05-16      Cake            Cash         12.0
2       2023-07-19    Cookie     Credit Card          4.0
3       2023-04-27     Salad             NaN         10.0
4       2023-06-11    Coffee  Digital Wallet          4.0
5       2023-03-31  Smoothie     Credit Card         20.0
6       2023-10-06       NaN             NaN          9.0
7       2023-10-28  Sandwich            Cash         16.0
8      

In [11]:
#  Outlier detection: use IQR method or Z-score to identify outliers in numeric columns; decide and document whether to cap, remove, or retain each 

In [12]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("dirty_cafe_sales.csv")

# Standardize text placeholders and convert to numeric for outlier analysis
df["Quantity"] = pd.to_numeric(
    df["Quantity"].replace(["UNKNOWN", "ERROR"], np.nan), errors="coerce"
)
df["Price Per Unit"] = pd.to_numeric(
    df["Price Per Unit"].replace(["UNKNOWN", "ERROR"], np.nan), errors="coerce"
)

# Recalculate Total Spent to ensure mathematical accuracy before checking outliers
df["Total Spent"] = df["Quantity"] * df["Price Per Unit"]

print("=== OUTLIER DETECTION (IQR METHOD) ===\n")


def detect_iqr_bounds(series):
    """Calculates lower and upper bounds using the 1.5 * IQR rule."""
    clean_series = series.dropna()
    q1 = clean_series.quantile(0.25)
    q3 = clean_series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return lower_bound, upper_bound, q1, q3


# Analyze numeric columns
numeric_cols = ["Quantity", "Price Per Unit", "Total Spent"]

for col in numeric_cols:
    lower_b, upper_b, q1, q3 = detect_iqr_bounds(df[col])

    # Identify outliers
    outliers_below = df[df[col] < lower_b][col]
    outliers_above = df[df[col] > upper_b][col]
    total_outliers = len(outliers_below) + len(outliers_above)

    print(f"--- Column: {col} ---")
    print(f"  Q1 (25th pct): {q1} | Q3 (75th pct): {q3}")
    print(f"  IQR Bounds:    Lower = {lower_b} to Upper = {upper_b}")
    print(f"  Outliers Found: {total_outliers} rows")
    if total_outliers > 0:
        print(f"  Min Outlier:   {df[col].loc[outliers_below.index.union(outliers_above.index)].min()}")
        print(f"  Max Outlier:   {df[col].loc[outliers_below.index.union(outliers_above.index)].max()}")
    print(f"  Operational Strategy: RETAIN ALL VALID ENTRIES\n")

print("Pipeline Action: No rows modified or capped during outlier phase.")

=== OUTLIER DETECTION (IQR METHOD) ===

--- Column: Quantity ---
  Q1 (25th pct): 2.0 | Q3 (75th pct): 4.0
  IQR Bounds:    Lower = -1.0 to Upper = 7.0
  Outliers Found: 0 rows
  Operational Strategy: RETAIN ALL VALID ENTRIES

--- Column: Price Per Unit ---
  Q1 (25th pct): 2.0 | Q3 (75th pct): 4.0
  IQR Bounds:    Lower = -1.0 to Upper = 7.0
  Outliers Found: 0 rows
  Operational Strategy: RETAIN ALL VALID ENTRIES

--- Column: Total Spent ---
  Q1 (25th pct): 4.0 | Q3 (75th pct): 12.0
  IQR Bounds:    Lower = -8.0 to Upper = 24.0
  Outliers Found: 240 rows
  Min Outlier:   25.0
  Max Outlier:   25.0
  Operational Strategy: RETAIN ALL VALID ENTRIES

Pipeline Action: No rows modified or capped during outlier phase.


In [13]:
#  Data type correction: ensure all columns have the correct dtype (e.g., dates as datetime, IDs as string, monetary values as float) 

In [14]:
import pandas as pd

# Load dataset
df = pd.read_csv("dirty_cafe_sales.csv")

print("=== BEGIN DATA TYPE CORRECTION PIPELINE ===\n")

# --- Step 1: Pre-coercion Cleaning (Isolating Numeric & Temporal Strings) ---
# Stripping hidden white spaces and forcing custom text placeholders to true structural NaNs
invalid_tokens = ["UNKNOWN", "ERROR", "NAN", "NULL", ""]
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].mask(df[col].str.upper().isin(invalid_tokens))

# --- Step 2: Schema Coercion & Casting ---

# Core Text / Identifiers -> High-efficiency String representation
df["Transaction ID"] = df["Transaction ID"].astype("string")

# Quantitative Continuous / Monetary Metrics -> Float64
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")

# Quantitative Discrete -> Int32 (Filling missing quantities with median first to allow integer cast)
median_qty = (
    pd.to_numeric(df["Quantity"], errors="coerce").median()
)  # Defaults to 2.0
df["Quantity"] = (
    pd.to_numeric(df["Quantity"], errors="coerce").fillna(median_qty).astype("int32")
)

# Programmatically recalculate 'Total Spent' to fix broken calculations before final float cast
df["Total Spent"] = (df["Quantity"] * df["Price Per Unit"]).astype("float64")

# Low-cardinality nominal values -> Category Type (Optimizes memory footprint)
df["Item"] = df["Item"].astype("category")
df["Payment Method"] = df["Payment Method"].fillna("Unknown").astype("category")
df["Location"] = df["Location"].fillna("Unknown").astype("category")

# Temporal Metrics -> Datetime64 Type
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

print("=== FINAL STRUCTURAL SCHEMA VALIDATION ===")
print(df.dtypes)
print("-" * 45)
print("Memory Usage Optimization Profile:")
print(df.info(memory_usage="deep"))

=== BEGIN DATA TYPE CORRECTION PIPELINE ===

=== FINAL STRUCTURAL SCHEMA VALIDATION ===
Transaction ID              string
Item                      category
Quantity                     int32
Price Per Unit             float64
Total Spent                float64
Payment Method            category
Location                  category
Transaction Date    datetime64[us]
dtype: object
---------------------------------------------
Memory Usage Optimization Profile:
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  string        
 1   Item              9031 non-null   category      
 2   Quantity          10000 non-null  int32         
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9467 non-null   float64       
 5   Payment Method    10000 non-null  category      
 6   Loca

In [15]:
#  Produce a "before vs. after" summary table: null count, duplicate count, row count, and dtype accuracy — before and after cleaning 

In [19]:
import numpy as np
import pandas as pd

# Load raw and execute pipeline to get clean state
df_raw = pd.read_csv("dirty_cafe_sales.csv")

# Quick execution of our cleaning pipeline to derive the cleaned dataframe
df_clean = df_raw.copy()
invalid_tokens = ["UNKNOWN", "ERROR", "NAN", "NULL", ""]

for col in df_clean.columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()
    df_clean[col] = df_clean[col].mask(
        df_clean[col].str.upper().isin(invalid_tokens)
    )

# Menu mapping for imputation
menu_map = {1.0: "Cookie", 1.5: "Tea", 2.0: "Coffee", 3.0: "Cake", 4.0: "Sandwich", 5.0: "Salad"}  # Cake/Juice simplified for demonstration
price_map = {"Cookie": 1.0, "Tea": 1.5, "Coffee": 2.0, "Cake": 3.0, "Juice": 3.0, "Sandwich": 4.0, "Smoothie": 4.0, "Salad": 5.0}

df_clean["Price Per Unit"] = pd.to_numeric(
    df_clean["Price Per Unit"], errors="coerce"
)
df_clean["Quantity"] = (
    pd.to_numeric(df_clean["Quantity"], errors="coerce").fillna(2).astype(int)
)
df_clean["Item"] = df_clean["Item"].fillna(
    df_clean["Price Per Unit"].map(menu_map)
)
df_clean["Price Per Unit"] = df_clean["Price Per Unit"].fillna(
    df_clean["Item"].map(price_map)
)
df_clean["Total Spent"] = df_clean["Quantity"] * df_clean["Price Per Unit"]
df_clean["Payment Method"] = df_clean["Payment Method"].fillna("Unknown")
df_clean["Location"] = df_clean["Location"].fillna("Unknown")
df_clean["Transaction Date"] = pd.to_datetime(
    df_clean["Transaction Date"], errors="coerce"
)

# Apply listwise drops for remaining unrecoverable target dates/items
df_clean = df_clean.dropna(subset=["Item", "Transaction Date"])

# Cast to optimized schema
schema = {
    "Transaction ID": "string",
    "Item": "category",
    "Quantity": "int32",
    "Price Per Unit": "float64",
    "Total Spent": "float64",
    "Payment Method": "category",
    "Location": "category",
}
df_clean = df_clean.astype(schema)

print("=== PIPELINE DATA QUALITY VALIDATION METRICS ===")
print(f"Raw Row Count: {len(df_raw)} | Cleaned Row Count: {len(df_clean)}")
print(f"Raw Missing Values Count: {df_raw.isna().sum().sum()}")
print(f"Cleaned Missing Values Count: {df_clean.isna().sum().sum()}")

=== PIPELINE DATA QUALITY VALIDATION METRICS ===
Raw Row Count: 10000 | Cleaned Row Count: 9489
Raw Missing Values Count: 6826
Cleaned Missing Values Count: 0


In [20]:
# Save the cleaned DataFrame to a CSV file
df.to_csv("clean_cafe_sales.csv", index=False)

In [21]:
import os

print("Your file is saved here:")
print(os.path.abspath("clean_cafe_sales.csv"))

Your file is saved here:
C:\Users\kinid\clean_cafe_sales.csv
